In [113]:
import numpy as np
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle
import gzip

In [57]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [30]:
df_fake = pd.read_csv('/content/drive/MyDrive/Fake_News_Detection/Fake.csv')
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [31]:
df_true = pd.read_csv('/content/drive/MyDrive/Fake_News_Detection/True.csv')
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [32]:
df_true['class'] = 1
df_fake['class'] = 0

In [33]:
df_true.shape

(21417, 5)

In [34]:
df = pd.concat([df_true,df_fake],axis=0)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44898 entries, 0 to 23480
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   class    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 2.1+ MB


In [36]:
df = df[df['text'] != ' ']

In [37]:
df = df.sample(frac=1).reset_index()

In [38]:
model_df = df.drop(['index','title','subject','date'],axis=1)

In [47]:
def normalise(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_14174/608854905.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipykernel_14174/608854905.py:5: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
/tmp/ipykernel_14174/608854905.py:9: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [49]:
model_df['text'] = model_df['text'].apply(normalise)

In [79]:
def stem_text(text):
    stop_words = list(stopwords.words('english'))
    word_token = word_tokenize(text.lower())
    filter_words = [x for x in word_token if x not in stop_words]
    snowball_stemmer = SnowballStemmer(language='english')
    snowball_stems = [snowball_stemmer.stem(word) for word in filter_words]
    new_sent = ' '.join(snowball_stems)
    return new_sent

In [81]:
model_df['text'] = model_df['text'].apply(stem_text)

In [82]:
model_df.head()

,text,class
0,ankara reuter turkey russia iran hold summit d...,1
1,budapest reuter rap knuckl eu top court end hu...,1
2,washington reuter obama administr thursday unv...,1
3,word yet mayor develop plan help christian ref...,0
4,osama bin laden bodyguard terrorist releas uae...,0


In [94]:
X = model_df['text']
y = model_df['class']
X_train,X_test,y_train,y_test = train_test_split(X,y, train_size=0.7,random_state=42)

In [95]:
vectorisation = TfidfVectorizer()
X_train_v = vectorisation.fit_transform(X_train)

In [96]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_v,y_train)

RandomForestClassifier(random_state=42)

In [99]:
Y_train_pred =model.predict(X_train_v)

In [101]:
from sklearn.metrics import classification_report
print(classification_report(y_train,Y_train_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     16037
           1       1.00      1.00      1.00     14952

    accuracy                           1.00     30989
   macro avg       1.00      1.00      1.00     30989
weighted avg       1.00      1.00      1.00     30989



In [105]:
X_test_v = vectorisation.transform(X_test)

In [106]:
y_test_pred = model.predict(X_test_v)

In [107]:
print(classification_report(y_test_pred,y_test))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      6829
           1       0.99      0.99      0.99      6453

    accuracy                           0.99     13282
   macro avg       0.99      0.99      0.99     13282
weighted avg       0.99      0.99      0.99     13282



In [114]:
with gzip.open("model.pkl.gz", "wb") as f:
    pickle.dump(model, f)

In [109]:
pickle.dump(vectorisation,open("vect.pkl","wb"))

In [110]:
!pip install streamlit -q
!wget -q -O - ipv4.icanhazip.com

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 67.7 MB/s eta 0:00:00
35.199.38.116


In [ ]:
! streamlit run /content/drive/MyDrive/Fake_News_Detection/app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://slick-cougars-travel.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.199.38.116:8501

2026-03-27 08:54:28.909 Script compilation error
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_runner.py", line 591, in _run_script
    code = self._script_cache.get_bytecode(script_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/script_cache.py", line 72, in get_bytecode
    filebody = magic.add_magic(filebody, script_path)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/streamlit/runtime/scriptrunner/magic.py", line 45, in add_magic
    tree = ast.parse(code, script_path, "exec")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi